Train the Model

In [ ]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

log_model.fit(X_train_scaled,y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

Predict Proba

In [ ]:
y_prob = log_model.predict_proba(X_test_scaled)[:,1]

Evaluation

ROC AUC

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test,y_prob)

print(auc)

0.7610638626874903


Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = log_model.predict(X_test)

cm = confusion_matrix(y_test,y_pred)

print(cm)

[[ 404 1481]
 [ 144  971]]


e:\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


Classification report

In [ ]:
from sklearn.metrics import classification_report

cr = classification_report(y_test,y_pred)

print(cr)

              precision    recall  f1-score   support

           0       0.74      0.21      0.33      1885
           1       0.40      0.87      0.54      1115

    accuracy                           0.46      3000
   macro avg       0.57      0.54      0.44      3000
weighted avg       0.61      0.46      0.41      3000



KS

In [ ]:
ks_df = pd.DataFrame({
    "actual": y_test,
    "prob": y_prob
})

In [ ]:
ks_df.head()

,actual,prob
1185,0,0.180940
3819,0,0.259672
3444,0,0.377222
7297,0,0.248469
7618,1,0.901765


In [ ]:
ks_df = ks_df.sort_values("prob", ascending=False)

In [ ]:
ks_df.head()

,actual,prob
6790,1,0.998834
5222,1,0.998753
9473,1,0.997735
3182,1,0.997356
6795,1,0.997132


Create Decile

In [ ]:
ks_df["decile"] = pd.qcut(
    ks_df['prob'],
    10,
    labels=False,
    duplicates='drop'
)

In [ ]:
ks_table = ks_df.groupby("decile").agg(
    loans=("actual","count"),
    bad=("actual","sum")
)

ks_table["good"] = ks_table["loans"] - ks_table["bad"]
ks_table["bad_rate"] = ks_table["bad"] / ks_table["loans"]

In [ ]:
ks_table = ks_table.sort_index(ascending=False)

ks_table["cum_bad"] = ks_table["bad"].cumsum() / ks_table["bad"].sum()
ks_table["cum_good"] = ks_table["good"].cumsum() / ks_table["good"].sum()

ks_table["ks"] = abs(ks_table["cum_bad"] - ks_table["cum_good"])

In [ ]:
ks_value = ks_table["ks"].max()

print("KS:", ks_value)

KS: 0.4010895552568662


In [ ]:
ks_table

,loans,bad,good,bad_rate,cum_bad,cum_good,ks
decile,,,,,,,
9,300,248,52,0.826667,0.222422,0.027586,0.194835
8,300,190,110,0.633333,0.392825,0.085942,0.306883
7,300,155,145,0.516667,0.531839,0.162865,0.368974
6,300,134,166,0.446667,0.652018,0.250928,0.401090
5,300,89,211,0.296667,0.731839,0.362865,0.368974
4,300,96,204,0.320000,0.817937,0.471088,0.346850
3,300,64,236,0.213333,0.875336,0.596286,0.279050
2,300,60,240,0.200000,0.929148,0.723607,0.205541
1,300,41,259,0.136667,0.965919,0.861008,0.104911


Check IV

In [ ]:
import scorecardpy as sc

In [ ]:
iv = sc.iv(df,y="FPD_FLAG")

e:\anaconda3\Lib\site-packages\scorecardpy\condition_fun.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1 1 1 ... 0 1 1]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  dat.loc[:,y] = dat[y].apply(lambda x: x if pd.isnull(x) else int(x)) #dat[y].astype(int)


In [ ]:
print(iv)

              variable  info_value
9          past_dpd_30    0.521664
4            foir_band    0.314620
14       tenure_months    0.110192
12     employment_type    0.085609
10          score_band    0.054640
1          loan_amount    0.016315
13                 age    0.016018
2         bureau_score    0.015385
17      monthly_income    0.014235
3       loan_to_income    0.010448
11       interest_rate    0.010362
5   credit_utilization    0.010362
16             loan_id    0.010362
0                  emi    0.010362
15                foir    0.010362
6         years_at_job    0.006798
7     num_active_loans    0.004531
8            city_tier    0.000386


WOE

Remove unique identifiers before WOE

In [ ]:
df.columns

Index(['loan_id', 'age', 'monthly_income', 'employment_type', 'years_at_job',
       'city_tier', 'bureau_score', 'num_active_loans', 'past_dpd_30',
       'credit_utilization', 'loan_amount', 'interest_rate', 'tenure_months',
       'emi', 'foir', 'loan_to_income', 'FPD_FLAG', 'score_band', 'foir_band'],
      dtype='object')

In [ ]:
df_model = df.drop(columns="loan_id")

converts categorical variables to string/object

In [ ]:
df_model = df_model.copy()

for col in df_model.select_dtypes(include=["category"]).columns:
    df_model[col] = df_model[col].astype(str)

In [ ]:
bins = sc.woebin(df_model,y="FPD_FLAG")

df_woe = sc.woebin_ply(df_model,bins)

[INFO] creating woe binning ...


e:\anaconda3\Lib\site-packages\scorecardpy\condition_fun.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1 1 1 ... 0 1 1]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  dat.loc[:,y] = dat[y].apply(lambda x: x if pd.isnull(x) else int(x)) #dat[y].astype(int)
e:\anaconda3\Lib\site-packages\scorecardpy\condition_fun.py:40: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  datetime_cols = dat.apply(pd.to_numeric,errors='ignore').select_dtypes(object).apply(pd.to_datetime,errors='ignore').select_dtypes('datetime64').columns.tolist()
e:\anaconda3\Lib\site-packages\scorecardpy\condition_fun.py:40: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  dat

Binning on 10000 rows and 18 columns in 00:00:10
[INFO] converting into woe values ...


In [ ]:
df_woe.head()

,FPD_FLAG,emi_woe,loan_amount_woe,age_woe,bureau_score_woe,loan_to_income_woe,foir_band_woe,credit_utilization_woe,tenure_months_woe,years_at_job_woe,num_active_loans_woe,city_tier_woe,past_dpd_30_woe,foir_woe,score_band_woe,interest_rate_woe,monthly_income_woe,employment_type_woe
0,1,0.009422,0.115681,0.103099,-0.075010,-0.113012,-0.392700,0.435858,-0.323491,0.020635,0.071219,0.012484,-0.554916,-0.461452,0.027988,0.004996,-0.310789,-0.203384
1,1,0.009422,-0.474020,0.012397,0.294357,-0.511816,-0.392700,-0.189062,0.536926,0.020635,0.074514,0.012484,0.491881,-0.197354,0.387647,0.008240,-0.023515,0.423939
2,1,0.009422,0.115681,-0.039850,-0.349325,0.091211,-0.048608,-0.358801,-0.127935,0.020635,0.013313,0.012484,-0.554916,0.122691,-0.307215,0.004996,-0.023515,0.423939
3,1,0.607586,0.115681,-0.050082,0.294357,0.091211,0.575256,-0.189062,0.536926,-0.027635,0.013313,0.007301,0.491881,0.699366,0.387647,0.008240,-0.023515,-0.203384
4,0,0.009422,0.429621,0.012397,-0.075010,0.302873,0.255230,-0.358801,-0.127935,-0.127631,0.071219,0.012484,-0.554916,0.122691,0.027988,0.008081,-0.023515,-0.203384


In [ ]:
X = df_woe.drop(columns="FPD_FLAG")
y = df_woe["FPD_FLAG"]

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42,stratify=y)

In [ ]:
log_model.fit(X_train,y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [ ]:
y_pred_prob = log_model.predict_proba(X_test)[:,1]

In [ ]:
auc = roc_auc_score(y_test,y_pred_prob)

print(auc)

0.7667623794173972


There is no much improvement after WOE so moving to another models

Random Forest

In [ ]:
X = df_model.drop(columns=["FPD_FLAG"])
y = df_model["FPD_FLAG"]

In [ ]:
X = pd.get_dummies(X,drop_first=True)

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,stratify=y,random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42
)

rf_model.fit(X_train,y_train)

RandomForestClassifier(max_depth=6, n_estimators=300, random_state=42)

In [ ]:
rf_pred_prob = rf_model.predict_proba(X_test)[:,1]

In [ ]:
auc = roc_auc_score(y_test,rf_pred_prob)
print(auc)

0.7524473361801336


In [ ]:
rf_pred = rf_model.predict(X_test)

In [ ]:
confusion_matrix(y_test,rf_pred)

array([[1686,  199],
       [ 669,  446]], dtype=int64)

In [ ]:
cr = classification_report(y_test,rf_pred)

print(cr)

              precision    recall  f1-score   support

           0       0.72      0.89      0.80      1885
           1       0.69      0.40      0.51      1115

    accuracy                           0.71      3000
   macro avg       0.70      0.65      0.65      3000
weighted avg       0.71      0.71      0.69      3000



Random forest is performing better than Logistic regression so moving to next model

XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators = 400,
    max_depth = 4,
    learning_rate = 0.05,
    subsample = 0.8,
    colsample_bytree = 0.8,
    random_state = 42
)

xgb_model.fit(X_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
xgb_pred_prob = xgb_model.predict_proba(X_test)[:,1]

In [ ]:
auc = roc_auc_score(y_test,xgb_pred_prob)

print(auc)

0.7520943012453758


In [ ]:
xgb_pred = xgb_model.predict(X_test)

In [ ]:
confusion_matrix(y_test,xgb_pred)

array([[1616,  269],
       [ 571,  544]], dtype=int64)

In [ ]:
cr = classification_report(y_test,xgb_pred)

print(cr)

              precision    recall  f1-score   support

           0       0.74      0.86      0.79      1885
           1       0.67      0.49      0.56      1115

    accuracy                           0.72      3000
   macro avg       0.70      0.67      0.68      3000
weighted avg       0.71      0.72      0.71      3000



Xgboost has best performance compared to other models 

Save Log model

In [ ]:
import joblib

joblib.dump(xgb_model,'xgb_model.pkl')

['xgb_model.pkl']